# Experiment 5 - Part 1: Keyword Search with TF-IDF and How to Measure Retrieval

**Course:** HAAI302COM - Artificial Intelligence Laboratory

Search is the core of RAG (Experiment 4), recommendation and question answering. This experiment compares two families of search:

| | Keyword search (this notebook) | Semantic search (notebooks 02 and 03) |
|---|---|---|
| Idea | Match the words of the query against the words of each document | Compare the *meaning* of the query and documents using embedding vectors |
| Tool | `TfidfVectorizer` (scikit-learn) | SentenceTransformers + FAISS |
| Weak spot | Different words, same meaning | Rare exact terms, speed at large scale |

To compare fairly we need a benchmark with the right answers. We use **SciFact** from the BEIR benchmark: 5,183 scientific abstracts, and 300 short scientific claims, each paired with the abstract(s) that support or refute it. A good search engine should return that abstract near the top when given the claim.

## 0. Setup and data

`ir_utils.py` (in this folder) loads SciFact and holds the metric function we build in section 3. Run the install line on Colab.

In [1]:
# !pip install -q scikit-learn sentence-transformers faiss-cpu datasets pandas matplotlib
import os, time, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

import ir_utils
data = ir_utils.load_scifact()
print({k: len(v) for k, v in data.items()})

{'doc_ids': 5183, 'titles': 5183, 'texts': 5183, 'q_ids': 300, 'q_texts': 300, 'relevant': 300}


In [2]:
# the corpus: one row per abstract
docs = pd.DataFrame({"doc_id": data["doc_ids"], "title": data["titles"], "words": [len(t.split()) for t in data["texts"]]})
print(docs.words.describe().round(0).to_string())
docs.head(3)

count    5183.0
mean      202.0
std        86.0
min        26.0
25%       147.0
50%       192.0
75%       247.0
max      1524.0


,doc_id,title,words
0,4983,Microstructural development of human newborn cerebral white matter assessed in vivo by...,278
1,5836,Induction of myelodysplasia by myeloid-derived suppressor cells.,201
2,7912,"BC1 RNA, the transcript from a master gene for ID element amplification, is able to pr...",180


**What to notice:** abstracts average about 200 words (the longest has over 1,500). This matters later because embedding models have a maximum input length.

Next, the queries and their relevance judgments. A **qrel** is one fact: "document D is relevant to query Q".

In [3]:
n_rel = pd.Series([len(r) for r in data["relevant"]])
print("queries:", len(data["q_ids"]), "| relevant documents per query:", n_rel.value_counts().sort_index().to_dict())
i = 3
print("\nCLAIM   :", data["q_texts"][i])
for d in data["relevant"][i]:
    print("RELEVANT:", data["titles"][d], "\n          ", data["texts"][d][:230], "...")

queries: 300 | relevant documents per query: {1: 277, 2: 14, 3: 4, 4: 3, 5: 2}

CLAIM   : 5% of perinatal mortality is due to low birth weight.
RELEVANT: Estimates of global prevalence of childhood underweight in 1990 and 2015. 
           CONTEXT One key target of the United Nations Millennium Development goals is to reduce the prevalence of underweight among children younger than 5 years by half between 1990 and 2015. OBJECTIVE To estimate trends in childhood unde ...


**What to notice:** 277 of the 300 queries have exactly one relevant abstract among 5,183, so this is a needle-in-a-haystack task. Look at the example: the claim is about perinatal mortality and low birth weight, while the relevant abstract is about childhood underweight prevalence and shares almost no exact words with it. That vocabulary gap is what separates keyword from semantic search.

## 1. TF-IDF in four steps

`TfidfVectorizer` converts each document into a vector with one number per vocabulary word. The number is **term frequency x inverse document frequency**: high when a word is frequent in this document but rare across the corpus.

In plain words: `weight = (how often the word appears here) x log(total documents / documents containing the word)`. The word "the" appears everywhere so gets a weight near zero; "tamoxifen" appears in few documents so gets a high weight.

Key arguments: `lowercase`, `stop_words`, `ngram_range`, `min_df` / `max_df` (ignore rare / too common words), `sublinear_tf` (use `1 + log(tf)` so repeating a word 20 times is not 20 times better), `norm` (length-normalise each vector to 1).

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

doc_texts = [t + ". " + x for t, x in zip(data["titles"], data["texts"])]      # title + abstract
tfidf = TfidfVectorizer()                                                       # all defaults
X = tfidf.fit_transform(doc_texts)                                              # sparse matrix, docs x vocabulary
print("matrix shape:", X.shape, "| type:", type(X).__name__)
print("non-zero entries:", X.nnz, "| density:", f"{X.nnz / (X.shape[0] * X.shape[1]):.4%}")

matrix shape: (5183, 35899) | type: csr_matrix
non-zero entries: 613302 | density: 0.3296%


**What to notice:** the matrix is 5,183 documents by about 36,000 distinct words, but only about 0.33 percent of the cells are non-zero. This is why TF-IDF uses a *sparse* matrix: storing the zeros would waste over 99 percent of the memory.

Let us look inside: the vocabulary and the inverse document frequency (`idf_`) of each word.

In [5]:
vocab = pd.DataFrame({"word": tfidf.get_feature_names_out(), "idf": tfidf.idf_,
                      "doc_freq": np.asarray((X > 0).sum(axis=0)).ravel()})
print("rarest words are those in 1 document; idf range:", vocab.idf.min().round(2), "to", vocab.idf.max().round(2))
pd.concat([vocab.sort_values("idf").head(6).reset_index(drop=True),
           vocab.sort_values("idf").tail(6).reset_index(drop=True)], axis=1, keys=["most common", "rarest"])

rarest words are those in 1 document; idf range: 1.0 to 8.86


most common                     rarest                   
         word       idf doc_freq    word       idf doc_freq
0          of  1.001931     5173    0021  8.860185        1
1         the  1.004640     5159   00197  8.860185        1
2         and  1.004834     5158   00005  8.860185        1
3          in  1.018692     5087      ₄₂  8.860185        1
4          to  1.056750     4897  ﬁnding  8.860185        1
5        that  1.218621     4165   00002  8.860185        1

**What to notice:** the lowest idf values (about 1.0) belong to "of", "the", "and", "in", found in almost every abstract; they barely influence a ranking. The highest idf (about 8.9) belongs to tokens that occur in a single abstract, and the list shows how messy real text is: numbers (`0021`), a subscript character and `ﬁnding` spelled with a ligature (the same PDF problem as in Experiment 4). Such tokens never match a query but inflate the vocabulary.

Now one document as a vector: which words carry its weight?

In [6]:
row = X[0].tocoo()
top = pd.DataFrame({"word": tfidf.get_feature_names_out()[row.col], "tfidf": row.data}).sort_values("tfidf", ascending=False)
print(data["titles"][0][:90])
print("words in this document:", len(top), "| vector length (L2 norm):", round(float(np.sqrt((row.data ** 2).sum())), 3))
top.head(8).reset_index(drop=True)

Microstructural development of human newborn cerebral white matter assessed in vivo by dif
words in this document: 123 | vector length (L2 norm): 1.0


,word,tfidf
0,diffusion,0.386971
1,matter,0.386971
2,white,0.337754
3,microm2,0.235821
4,infants,0.226852
5,term,0.205632
6,tensor,0.186001
7,cerebral,0.185746


**What to notice:** the heaviest words are specific terms of this abstract (diffusion, matter, white, infants, tensor, cerebral), not generic words. The vector length is 1.0, because `norm="l2"` is the default; that makes a plain dot product between two vectors equal their cosine similarity.

## 2. Searching

To search we turn the query into a vector with the *same fitted vocabulary* (`transform`, never `fit_transform`), compute the cosine similarity to every document, and sort. Words in the query that are not in the vocabulary are simply ignored.

In [7]:
from sklearn.metrics.pairwise import linear_kernel

def tfidf_rank(vectorizer, X, queries, k=100):
    Q = vectorizer.transform(queries)                       # reuse the fitted vocabulary
    S = linear_kernel(Q, X)                                 # dot product = cosine (rows are L2-normalised)
    return np.argsort(-S, axis=1)[:, :k], S

ranked, scores = tfidf_rank(tfidf, X, data["q_texts"])
print("score matrix:", scores.shape, "(queries x documents)")

score matrix: (300, 5183) (queries x documents)


In [8]:
i = 3
print("CLAIM:", data["q_texts"][i], "\n")
pd.DataFrame({"rank": range(1, 6), "score": scores[i, ranked[i, :5]].round(3),
              "relevant": [d in data["relevant"][i] for d in ranked[i, :5]],
              "title": [data["titles"][d][:80] for d in ranked[i, :5]]})

CLAIM: 5% of perinatal mortality is due to low birth weight. 



,rank,score,relevant,title
0,1,0.372,False,Perinatal mortality in rural China: retrospective cohort study.
1,2,0.358,False,Intrauterine environments and breast cancer risk: meta-analysis and systematic r
2,3,0.269,False,Birth weight and mammographic density among postmenopausal women in Sweden.
3,4,0.251,False,Birth Size and Breast Cancer Risk: Re-analysis of Individual Participant Data fr
4,5,0.220,False,Prevalence of attempting weight loss and strategies for controlling weight.


**What to notice:** none of the five top results is the judged-relevant abstract. The engine matches the words "perinatal mortality", "birth weight" and returns papers about those words (including breast cancer studies that mention birth weight), but the relevant abstract talks about underweight children and shares no such words. One example proves nothing; we need numbers over all 300 queries.

## 3. Retrieval metrics

Four numbers are standard. For one query, let `rel` be the set of relevant documents and look at the top k results:

| Metric | In words |
|---|---|
| **Recall@k** | the fraction of the relevant documents that appear in the top k |
| **MRR@k** (mean reciprocal rank) | 1 divided by the rank of the first relevant result (0 if none in the top k); averaged over queries |
| **nDCG@k** | like a gain score where a relevant result at rank 1 is worth more than at rank 5 (weight 1 divided by log2(rank + 1)), divided by the best possible score |
| **Precision@k** | the fraction of the top k that are relevant |

We build them by hand on a toy ranking first. The list marks which of the top 5 results are relevant (1) or not (0), and 2 relevant documents exist in total.

In [9]:
hits = np.array([0, 1, 0, 0, 1])               # results ranked 1..5; 1 = relevant
n_relevant = 2                                 # relevant documents that exist for this query
ranks = np.arange(1, 6)
toy = pd.DataFrame({
    "rank": ranks, "hit": hits,
    "precision@k": hits.cumsum() / ranks,                       # relevant so far / results so far
    "recall@k": hits.cumsum() / n_relevant,                     # relevant so far / all relevant
    "discount": (1 / np.log2(ranks + 1)).round(3),              # 1 / log2(rank + 1)
})
toy

,rank,hit,precision@k,recall@k,discount
0,1,0,0.000000,0.0,1.000
1,2,1,0.500000,0.5,0.631
2,3,0,0.333333,0.5,0.500
3,4,0,0.250000,0.5,0.431
4,5,1,0.400000,1.0,0.387


In [10]:
rr = 1 / (np.argmax(hits) + 1)                                  # first relevant result is at rank 2
dcg = (hits / np.log2(ranks + 1)).sum()                         # gain of what we returned
idcg = (1 / np.log2(np.arange(1, n_relevant + 1) + 1)).sum()    # gain of the perfect ranking [1, 1, 0, 0, 0]
print(f"reciprocal rank = {rr:.3f} | DCG = {dcg:.3f} | ideal DCG = {idcg:.3f} | nDCG = {dcg / idcg:.3f}")

reciprocal rank = 0.500 | DCG = 1.018 | ideal DCG = 1.631 | nDCG = 0.624


**What to notice:** the first hit is at rank 2, so the reciprocal rank is 1/2. Recall reaches 0.5 at rank 2 and 1.0 at rank 5. The nDCG is 0.62 because a perfect ranking would put both relevant documents at ranks 1 and 2. We can cross-check with scikit-learn's implementation.

In [11]:
from sklearn.metrics import ndcg_score
# ndcg_score wants true relevance and predicted scores; give it descending scores in our rank order
print("sklearn ndcg:", round(ndcg_score([[0, 1, 0, 0, 1]], [[5, 4, 3, 2, 1]]), 3))

sklearn ndcg: 0.624


The match confirms our hand calculation. The same logic, looped over all queries, lives in `ir_utils.evaluate`. We run it on the TF-IDF ranking and check that it agrees with the toy numbers.

In [12]:
print("toy check:", ir_utils.evaluate([[10, 11, 12, 13, 14] + list(range(100, 195))], [{11, 14}]).to_dict())   # relevant at ranks 2 and 5
baseline, per_q = ir_utils.evaluate(ranked, data["relevant"], per_query=True)
baseline

toy check: {'recall@1': 0.0, 'recall@10': 1.0, 'recall@100': 1.0, 'mrr@10': 0.5, 'ndcg@10': 0.6241}


recall@1      0.4301
recall@10     0.7135
recall@100    0.8836
mrr@10        0.5437
ndcg@10       0.5780
dtype: float64

**What to notice:** the toy check gives recall@10 of 1.0, MRR of 0.5 and nDCG@10 of 0.62, equal to our manual numbers. For the real data, default TF-IDF puts a relevant abstract at rank 1 for 43 percent of the claims (`recall@1`), in the top 10 for 71 percent and in the top 100 for 88 percent. The gap between recall@10 and recall@100 says that for many claims the right abstract is retrieved, but ranked too low. Remember these numbers: they are the bar semantic search must beat.

## 4. Experiment: TF-IDF settings

Each argument of `TfidfVectorizer` changes the weights. We run the same 300 queries against eight settings and compare.

In [13]:
settings = {
    "default": {},
    "sublinear_tf": {"sublinear_tf": True},
    "stop_words=english": {"stop_words": "english"},
    "ngram_range=(1,2)": {"ngram_range": (1, 2)},
    "min_df=2": {"min_df": 2},
    "max_df=0.5": {"max_df": 0.5},
    "sublinear + stop + (1,2)": {"sublinear_tf": True, "stop_words": "english", "ngram_range": (1, 2)},
}
rows = []
for name, kw in settings.items():
    t0 = time.time()
    v = TfidfVectorizer(**kw)
    Xv = v.fit_transform(doc_texts)
    r, _ = tfidf_rank(v, Xv, data["q_texts"])
    m = ir_utils.evaluate(r, data["relevant"])
    rows.append({"setting": name, "vocab": Xv.shape[1], "nnz": Xv.nnz, **m[["recall@10", "mrr@10", "ndcg@10", "recall@100"]].to_dict(), "sec": round(time.time() - t0, 1)})
tfidf_results = pd.DataFrame(rows).set_index("setting")
tfidf_results

,vocab,nnz,recall@10,mrr@10,ndcg@10,recall@100,sec
setting,,,,,,,
default,35899,613302,0.7135,0.5437,0.5780,0.8836,1.5
sublinear_tf,35899,613302,0.7735,0.5897,0.6286,0.8916,1.6
stop_words=english,35617,466020,0.7235,0.5454,0.5811,0.8882,1.4
"ngram_range=(1,2)",477848,1585443,0.7319,0.5548,0.5924,0.8752,4.6
min_df=2,18744,596147,0.6685,0.5001,0.5349,0.8729,1.5
max_df=0.5,35885,556658,0.7168,0.5442,0.5790,0.8809,1.6
"sublinear + stop + (1,2)",511980,1106343,0.7790,0.6047,0.6418,0.8916,4.0


**What to notice:** read the `ndcg@10` column against `default` (0.578).

- `sublinear_tf=True` is the largest single gain (0.629). With the default, a word repeated many times in a long abstract keeps growing in weight; `1 + log(tf)` stops repetition from outweighing rare, distinctive words.
- `stop_words` and `max_df=0.5` change almost nothing (0.581 and 0.579): idf already shrinks common words.
- `ngram_range=(1,2)` gives a small gain (0.592) but the vocabulary grows from 36,000 to 478,000 features and `nnz` by 2.6 times, a poor trade on its own.
- `min_df=2` **hurts** (0.535). Words that occur in only one abstract are the most distinctive ones, and a rare word in a query is often the decisive match, so deleting them removes the answer.
- Combining sublinear, stop words and bigrams gives the best result (0.642).

Note that `recall@100` hardly moves (0.87 to 0.89) in any setting: about 11 percent of the relevant abstracts are not in the top 100 whatever we tune, because no query word occurs in them. Differences of 0.01 are small relative to the noise of a 300-query test.

## 5. Why does keyword search fail? Vocabulary mismatch

If a claim and its relevant abstract share no important words, TF-IDF has nothing to match. We measure that directly: for each query, the fraction of its distinct (non-stop) words that also occur in its relevant abstract. Then we compare queries the search got right with those it got wrong.

In [14]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
tok = TfidfVectorizer(stop_words="english").build_analyzer()          # same tokenizer, with stop words removed

def overlap(i):
    q = set(tok(data["q_texts"][i]))
    d = set().union(*[set(tok(doc_texts[j])) for j in data["relevant"][i]])
    return len(q & d) / max(len(q), 1)

per_q["word_overlap"] = [overlap(i) for i in range(len(data["q_ids"]))]
per_q["found_top10"] = per_q["recall@10"] > 0
per_q.groupby("found_top10")["word_overlap"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
found_top10,,,
False,80,0.32,0.25
True,220,0.59,0.60


**What to notice:** queries whose relevant abstract is found in the top 10 share clearly more of their words with it than the missed queries do. Search that depends on exact words behaves exactly like this: the more vocabulary a query shares with the answer, the better it works. Here are three missed queries with the biggest vocabulary gap.

In [15]:
missed = per_q[~per_q.found_top10].sort_values("word_overlap").head(3).index
for i in missed:
    d = next(iter(data["relevant"][i]))
    print(f"[{per_q.word_overlap[i]:.2f} overlap] CLAIM: {data['q_texts'][i]}\n             RELEVANT: {data['titles'][d][:110]}\n")

[0.00 overlap] CLAIM: 0-dimensional biomaterials show inductive properties.
             RELEVANT: New opportunities: the use of nanotechnologies to manipulate and track stem cells.

[0.00 overlap] CLAIM: In mice, P. chabaudi parasites are able to proliferate faster early in infection when inoculated at lower numbers than when inoculated at high numbers.
             RELEVANT: Partitioning regulatory mechanisms of within-host malaria dynamics using the effective propagation number.

[0.00 overlap] CLAIM: PPAR-RXRs are inhibited by PPAR ligands.
             RELEVANT: Unique response pathways are established by allosteric interactions among nuclear hormone receptors



**What to notice:** the claims are phrased as short natural-language statements, the relevant titles in technical vocabulary. A reader sees the connection by meaning; TF-IDF sees almost no shared words. Notebook 02 attacks exactly this problem with embeddings.

## Summary

| Idea | What we saw |
|---|---|
| TF-IDF vector | sparse (about 0.7 percent non-zero), weights are tf x idf, rare words dominate |
| Search | `transform` the query with the fitted vocabulary, cosine against the matrix, sort |
| Metrics | Recall@k (coverage), MRR (first hit), nDCG (rank-weighted); all computed by hand and cross-checked with scikit-learn |
| Tuning | weighting options move the scores a little; they cannot add words that are not there |
| Failure mode | missed queries share fewer words with their relevant abstract (vocabulary mismatch) |

**Next:** Notebook 02 encodes the same abstracts and claims as dense vectors with SentenceTransformers and compares on the same metrics.